<h2>🧪 Quick Sanity Check</h2>

<div style="padding: 10px 12px; border-left: 6px solid #10B981; background: #ECFDF5; border-radius: 10px;">
Optional mini-cell to confirm the kernel is alive ✅
</div>

<h1>⚙️ Setup</h1>

<div style="padding: 10px 12px; border-left: 6px solid #F59E0B; background: #d6b941ff; border-radius: 10px;">
Install + upgrade key libraries (UnsLoTh + TRL + Transformers).<br>
If you’re on Colab/Kaggle, this cell handles the heavy lifting 🧰
</div>

In [1]:
# %%capture
# import os, importlib.util
# !pip install --upgrade -qqq uv
# if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
#     try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
#     except: get_numpy = "numpy"; get_pil = "pillow"
#     !uv pip install -qqq \
#         "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
#         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#         "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#         git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# elif importlib.util.find_spec("unsloth") is None:
#     !uv pip install -qqq unsloth
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [2]:
# !pip install -U bitsandbytes

In [3]:
try:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

except:
    !uv pip install --system --no-index --find-links='/kaggle/input/datasets/barnobarno/unsloth-py-3-12/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

Using Python 3.12.12 environment at: /usr
  × No solution found when resolving dependencies:
  ╰─▶ Because torch==2.9.1 has no wheels with a matching Python ABI tag (e.g.,
      `cp312`) and xformers==0.0.33.post2 depends on torch==2.9.1, we can
      conclude that xformers==0.0.33.post2 cannot be used.
      And because only xformers{(platform_machine == 'AMD64' and 'linux'
      in sys_platform) or (platform_machine == 'x86_64' and 'linux' in
      sys_platform) or (platform_machine == 'AMD64' and sys_platform
      == 'win32') or (platform_machine == 'x86_64' and sys_platform
      == 'win32')}==0.0.33.post2 is available and unsloth==2026.1.2
      depends on xformers{(platform_machine == 'AMD64' and 'linux' in
      sys_platform) or (platform_machine == 'x86_64' and 'linux' in
      sys_platform) or (platform_machine == 'AMD64' and sys_platform
      == 'win32') or (platform_machine == 'x86_64' and sys_platform ==
      'win32')}==0.0.33.post2, we can conclude that unsloth==2026.1.

2026-02-21 15:16:18.716177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771686979.079654      44 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771686979.189654      44 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771686979.976539      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771686979.976572      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771686979.976576      44 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


<h1 style="color: #3B82F6;">🤖 Model</h1>

<h2 style="color: #3B82F6;">📥 Load Base Model</h2>

<div style="padding: 10px 12px; border-left: 6px solid #3B82F6; background: #EFF6FF; border-radius: 10px;">
Loads <b>unsloth/gpt-oss-20b</b> in 4-bit for VRAM-friendly training 🧊
</div>

In [4]:
print ("done")

done


In [5]:
# import kagglehub 
# path = kagglehub.model_download("barnobarno/gpt-oss-20b/transformers/unsloth")

# print("Path to model files:", path)

In [6]:
# %%capture
# import os, importlib.util
# !pip install --upgrade -qqq uv
# if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
#     try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
#     except: get_numpy = "numpy"; get_pil = "pillow"
#     !uv pip install -qqq \
#         "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
#         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#         "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#         git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# elif importlib.util.find_spec("unsloth") is None:
#     !uv pip install -qqq unsloth
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [7]:
import torch
max_seq_length = 16384   # Reduced from 2048 - speeds up generation significantly
 # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/models/barnobarno/gpt-oss-120b-bnb-4bit/transformers/unsloth/1" ,
    max_seq_length = max_seq_length,
    local_files_only =  True ,
    load_in_4bit=True , 
    dtype=None
)

==((====))==  Unsloth 2026.2.1: Fast Gpt_Oss patching. Transformers: 4.57.6.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

In [8]:
lora_rank = 16

In [9]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"


In [10]:
import re


<h2 style="color: #265ccfff;">🧩 LoRA Settings</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #654ed5ff; border-radius: 10px;">
Pick a LoRA rank. Higher = more capacity, slower + more memory 🔧
</div>

<h1 style="color: #1757ebff;">📚 Data</h1>

<h2>🧾 Load OlymMATH (en-hard)</h2>

<div style="padding: 10px 12px; border-left: 6px solid #06B6D4; background: #ECFEFF; border-radius: 10px;">
Pulls the dataset + previews it in a DataFrame 🗂️
</div>

In [11]:
# from datasets import load_dataset

# # Login using e.g. `huggingface-cli login` to access this dataset
# ds = load_dataset("RUC-AIBOX/OlymMATH", "en-hard")

In [12]:
import pandas as pd 
data = pd.read_csv("/kaggle/input/olymmath-hard/OlymMATH-EN-EASY.csv") # pd.DataFrame(ds['test'])
print(data.head())


                                             problem         answer  \
0  Given a non-negative integer sequence $\{a_n\}...            948   
1  Given that $AB$ is a diameter of circle $\odot...  4\sqrt{15}-14   
2  Calculate the value of $\sqrt{9+8\cos 20^{\cir...              3   
3  A sphere is circumscribed around tetrahedron $...   \frac{18}{5}   
4  Find the minimum value of $f(x) = \sum_{i=1}^{...      801730806   

         subject           unique_id  
0  Combinatorics  OlymMATH-EASY-0-EN  
1       Geometry  OlymMATH-EASY-1-EN  
2        Algebra  OlymMATH-EASY-2-EN  
3       Geometry  OlymMATH-EASY-3-EN  
4        Algebra  OlymMATH-EASY-4-EN  


<h1 style="color: #b0de4eff;">🧠 Prompting</h1>

<h2 style="color: #3e1492ff;">📝 System + User Prompt Template</h2>

<div style="padding: 10px 12px; border-left: 6px solid #22C55E; background: #F0FDF4; border-radius: 10px;">
Defines the instruction style: step-by-step reasoning + final integer inside <b>\\boxed{}</b> ✅
</div>

In [13]:
# Harmony-compatible prompt - model uses tool calling format naturally
SYSTEM_PROMPT = """You are an elite mathematical problem solver. Solve problems step-by-step using Python code when calculations are needed.

# Important:
- You will NOT see the output of your Python code - mentally trace what it would print
- Write code, predict its output yourself, then continue reasoning
- Do NOT repeat code blocks waiting for output - one well-written block is enough

# Output Format:
- Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}
- The final code block should print/verify your answer. Use print function explicitely if you want to print.

Think step-by-step. Write code once, predict the result, then give your final \\boxed{answer}."""

def format_prompt(question):
    return [
        {"role": "user", "content": f"{question}\n\nSolve this problem. Write Python code for calculations (you won't see the output - predict it yourself). Put your final integer answer in \\boxed{{answer}}."}
    ]

<h2 style="color: #e323a9ff;">🔌 LoRA: Add Adapters</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #F5F3FF; border-radius: 10px;">
Attaches LoRA modules to attention + MLP layers (fast, memory-friendly fine-tuning) 🧬
</div>

In [14]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapters added successfully!")

Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.model` require gradients
LoRA adapters added successfully!


<h2 style="color: #e323a9ff;">🧪 Build a Tiny Training Set</h2>

<div style="padding: 10px 12px; border-left: 6px solid #06B6D4; background: #081212; border-radius: 10px;">
Creates a small list of prompt/answer pairs for quick testing 🧫
</div>

In [15]:
from sklearn.model_selection import train_test_split

# Sample 10 datapoints, stratified by subject
# train_size=10 guarantees exactly 10 samples
# stratify=data['subject'] ensures the subject distribution matches the full dataset
sampled_data, _ = train_test_split(
    data, 
    train_size=2, 
    # stratify=data['subject'], 
    random_state=42  # Fixed seed for reproducibility
)

# Create training dataset from the sampled data
train_data = []
for _, row in sampled_data.iterrows():
    question = row['problem']
    answer = row['answer']  # The ground truth answer
    
    train_data.append({
        "prompt": format_prompt(question),
        "answer": str(answer),
        "reasoning_effort": "medium"
    })

print(f"Created dataset with {len(train_data)} problems")
if len(train_data) > 0:
    print(f"Sample question: {train_data[0]['prompt'][0]['content'][:200]}...")
    print(f"Sample answer: {train_data[0]['answer']}")

Created dataset with 2 problems
Sample question: Let $f(x): [0, 1] \rightarrow \mathbb{R}$, satisfying: (1) $f(\frac{x}{3}) = \frac{1}{2}f(x)$; (2) $f(1-x) = 1 - f(x)$; (3) $f(x) = \frac{1}{2} (x \in [\frac{1}{3}, \frac{2}{3}])$. If $n=2023$, find t...
Sample answer: \frac{3^{2023} + 3}{4}


<h1 style="color: #e323a9ff;">🧰 Parsing Utilities</h1>

<h2 style="color: rgb(141, 76, 76);">📦 NO NEED TO CHANGE <span style="color:#4F46E5;"><b></b></span></h2>

<div style="padding: 10px 12px; border-left: 6px solid #4F46E5; background: #08237b; border-radius: 10px;">
We score outputs by pulling the final value from <b>\\boxed{...}</b> 🔍
</div>

In [16]:
"""LOOKS OK HERE"""




# Answer extraction function
def extract_boxed_answer(text):
    """Extract the answer from \\boxed{} in the model output"""
    # Try to find \boxed{...}
    patterns = [
        r'\\boxed\{([^{}]*)\}',  # Simple \boxed{answer}
        r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}',  # Nested braces
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        if matches:
            # Return the last match (final answer)
            answer_str = matches[-1].strip()
            try:
                # Try to parse as number
                # Handle fractions, negatives, etc.
                answer_str = answer_str.replace(',', '')  # Remove commas
                if '/' in answer_str:
                    # Handle fractions
                    parts = answer_str.split('/')
                    return float(parts[0]) / float(parts[1])
                return float(answer_str)
            except:
                return None
    return None
"""The above function looks ok , keep as is"""



def parse_true_answer(answer_str):
    """Parse the ground truth answer to a number"""
    try:
        answer_str = str(answer_str).strip().replace(',', '')
        if '/' in answer_str:
            parts = answer_str.split('/')
            return float(parts[0]) / float(parts[1])
        return float(answer_str)
    except:
        return None

# Test extraction
test_text = "The answer is \\boxed{42}"
print(f"Test extraction: {extract_boxed_answer(test_text)}")

Test extraction: 42.0


<h1 style="color: #e323a9ff;">🎯 Rewards</h1>

<h2 style="color: #ffffffff;">✨ I SHOULD ADD A HARMONY REWARD + A TOKEN USAGE REWARD</h2>

<div style="padding: 10px 12px; border-left: 6px solid #F97316; background: #d392b3ff; border-radius: 10px;">
Two rewards:
<ul style="margin: 6px 0 0 18px;">
  <li><b>Format</b> reward for including <b>\\boxed{}</b> 🧾</li>
  <li><b>Answer</b> reward based on distance from the true value 🎯</li>
</ul>
</div>

In [17]:
"""THIS FUNCTION NEEDS TO BE TURNED INTO REWARD FUNCTION"""

# def repair_harmony_tokens(self, token_ids: list[int]) -> list[int]:
#         """
#         Robustly repairs malformed Harmony messages where <|message|> is skipped.
#         Specifically handles 'commentary' split into two tokens.
#         """
#         CHANNEL_TOKEN = 200005
#         MESSAGE_TOKEN = 200008
        
#         new_ids = []
#         i = 0
#         n = len(token_ids)
        
#         while i < n:
#             token = token_ids[i]
#             new_ids.append(token)
            
#             # Trigger only on <|channel|>
#             if token == CHANNEL_TOKEN:
#                 # Default channel length is 1 token
#                 channel_len = 1
                
#                 # Check for special case: 'commentary' split into two tokens
#                 if i + 2 < n:
#                     try:
#                         # Decode next two tokens
#                         chunk = token_ids[i+1 : i+3]
#                         decoded_chunk = self.tokenizer.decode(chunk).lower()
#                         if "commentary" in decoded_chunk:
#                             channel_len = 2
#                     except Exception:
#                         pass
                
#                 # Copy the channel tokens
#                 for _ in range(channel_len):
#                     if i + 1 < n:
#                         i += 1
#                         new_ids.append(token_ids[i])
                
#                 # Check if the NEXT token is <|message|>. If not, inject it.
#                 if i + 1 < n:
#                      if token_ids[i+1] != MESSAGE_TOKEN:
#                         new_ids.append(MESSAGE_TOKEN)
#                 else: 
#                      # End of sequence, better close with message token so parser doesn't fail
#                      new_ids.append(MESSAGE_TOKEN)
            
#             i += 1
            
#         return new_ids

'THIS FUNCTION NEEDS TO BE TURNED INTO REWARD FUNCTION'

In [18]:
# Reward Functions for RLVR
import re

def format_reward(completions, **kwargs):
    """Reward for proper formatting with \\boxed{}"""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Check if response contains \\boxed{}
        if "\\boxed{" in response:
            scores.append(0.5)  # Small bonus for correct format
        else:
            scores.append(-0.5)  # Penalty for missing format
    return scores








def answer_reward(completions, answer, **kwargs):
    """
    Main reward: negative distance from correct answer
    Reward = -1 * abs(TRUE - PREDICTED) / abs(TRUE) if TRUE != 0
    Reward = -1 * abs(PREDICTED) if TRUE == 0
    Reward = 0 if exactly correct
    """
    scores = []
    
    for completion, true_ans in zip(completions, answer):
        response = completion[0]["content"]
        predicted = extract_boxed_answer(response)
        true_value = parse_true_answer(true_ans)
        
        if predicted is None or true_value is None:
            # Can't parse answer - give penalty
            scores.append(-2.0)
            continue
        
        # Calculate distance-based reward
        if abs(predicted - true_value) < 1e-6:
            # Exactly correct!
            reward = 10.0  # Bonus for correct answer
        else:
            # Calculate relative error
            if abs(true_value) > 1e-6:
                relative_error = abs(true_value - predicted) / abs(true_value)
            else:
                relative_error = abs(predicted)
            
            # Negative reward based on error, capped at -2
            reward = -1.0 * min(relative_error, 2.0)
        
        scores.append(reward)
    
    return scores

# Test the reward function
test_completions = [[{"content": "The answer is \\boxed{42}"}]]
test_answers = ["42"]
print(f"Test reward (correct): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{40}"}]]
print(f"Test reward (close): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{0}"}]]
print(f"Test reward (far): {answer_reward(test_completions, test_answers)}")

Test reward (correct): [10.0]
Test reward (close): [-0.047619047619047616]
Test reward (far): [-1.0]


In [19]:
import re
import sys
import io
import math
import contextlib
import threading
import numpy as np
import sympy

# --- Helper 1: Extract ALL Python Code Blocks (Harmony + Markdown) ---
def extract_all_code_blocks(text):
    """
    Extract ALL code blocks from model output.
    Handles both:
    1. Harmony format: <|message|>CODE<|call|> or <|message|>CODE<|end|>
    2. Markdown format: ```python\nCODE\n```
    
    Returns list of code strings in order of appearance.
    """
    code_blocks = []
    
    # Pattern 1: Harmony format - code between <|message|> and <|call|> or <|end|>
    harmony_pattern = r'to=python\s*code\s*<\|message\|>(.*?)(?:<\|call\|>|<\|end\|>)'
    harmony_matches = re.findall(harmony_pattern, text, re.DOTALL | re.IGNORECASE)
    
    # Pattern 2: Simpler harmony - just <|message|>CODE<|call|> after "python" context
    simple_harmony = r'python\s*(?:code)?\s*<\|message\|>(.*?)(?:<\|call\|>|<\|end\|>)'
    simple_matches = re.findall(simple_harmony, text, re.DOTALL | re.IGNORECASE)
    
    # Pattern 3: Markdown code blocks (fallback)
    markdown_pattern = r'```python\n(.*?)\n```'
    markdown_matches = re.findall(markdown_pattern, text, re.DOTALL)
    
    # Combine all matches, prefer harmony if found
    if harmony_matches:
        code_blocks.extend([m.strip() for m in harmony_matches if m.strip()])
    elif simple_matches:
        code_blocks.extend([m.strip() for m in simple_matches if m.strip()])
    
    # Add markdown matches if any (may overlap, dedupe by content)
    seen = set(code_blocks)
    for m in markdown_matches:
        cleaned = m.strip()
        if cleaned and cleaned not in seen:
            code_blocks.append(cleaned)
            seen.add(cleaned)
    
    return code_blocks


# --- Helper 2: Extract boxed answer from FULL model text output ---
def extract_boxed_from_text(text):
    """
    Extract the \\boxed{} answer from the model's full text output.
    The model writes \\boxed{} in a separate channel, NOT inside code.
    """
    if not text:
        return None
    
    # Various patterns for boxed answers in text
    patterns = [
        r'\\boxed\s*\{\s*([^{}]*)\s*\}',           # \boxed{42}
        r'\\\\boxed\s*\{\s*([^{}]*)\s*\}',         # \\boxed{42} (escaped)
        r'boxed\s*\{\s*([^{}]*)\s*\}',             # boxed{42} (no backslash)
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            # Take the LAST match (final answer)
            answer_str = matches[-1].strip().replace(',', '')
            try:
                # Handle fractions
                if '/' in answer_str:
                    parts = answer_str.split('/')
                    return float(parts[0]) / float(parts[1])
                return float(answer_str)
            except:
                pass
    
    return None


# --- Helper 3: Parse numerical output from code execution ---
def parse_code_output(output_str):
    """Extract a number from code execution output."""
    if not output_str:
        return None
    
    # Try last line first (most common: just print the result)
    lines = output_str.strip().split('\n')
    for line in reversed(lines):
        line = line.strip()
        try:
            # Handle tuples like (42, ...)
            if line.startswith('('):
                first_val = line.split(',')[0].strip('( ')
                return float(first_val)
            return float(line.replace(',', ''))
        except:
            continue
    return None


# --- Helper 4: Safe Execution with Shared Namespace ---
def create_safe_namespace():
    """Create a fresh isolated namespace with allowed libraries."""
    return {
        "math": math,
        "np": np,
        "numpy": np,
        "sympy": sympy,
        "itertools": __import__('itertools'),
        "collections": __import__('collections'),
        "functools": __import__('functools'),
        "random": __import__('random'),
        "print": print,
        "range": range,
        "len": len,
        "int": int,
        "float": float,
        "str": str,
        "list": list,
        "set": set,
        "dict": dict,
        "tuple": tuple,
        "abs": abs,
        "min": min,
        "max": max,
        "sum": sum,
        "pow": pow,
        "round": round,
        "enumerate": enumerate,
        "zip": zip,
        "sorted": sorted,
        "reversed": reversed,
        "map": map,
        "filter": filter,
        "all": all,
        "any": any,
        "True": True,
        "False": False,
        "None": None,
        "isinstance": isinstance,
        "type": type,
        "bool": bool,
        "divmod": divmod,
        "complex": complex,
        "hex": hex,
        "oct": oct,
        "bin": bin,
        "chr": chr,
        "ord": ord,
        "__builtins__": {
            "True": True,
            "False": False,
            "None": None,
            "isinstance": isinstance,
            "type": type,
            "len": len,
            "range": range,
            "print": print,
            "int": int,
            "float": float,
            "str": str,
            "bool": bool,
            "list": list,
            "dict": dict,
            "set": set,
            "tuple": tuple,
            "abs": abs,
            "min": min,
            "max": max,
            "sum": sum,
            "pow": pow,
            "round": round,
            "sorted": sorted,
            "reversed": reversed,
            "enumerate": enumerate,
            "zip": zip,
            "map": map,
            "filter": filter,
            "all": all,
            "any": any,
            "divmod": divmod,
            "complex": complex,
            "hex": hex,
            "oct": oct,
            "bin": bin,
            "chr": chr,
            "ord": ord,
        },
    }


def execute_in_namespace(code, namespace, timeout=5):
    """
    Execute code in given namespace, capturing stdout.
    Returns (output, success) - namespace modified in place.
    Reduced timeout to 5s to speed up training.
    """
    result = {"output": "", "success": False}
    
    def run_code():
        output_buffer = io.StringIO()
        try:
            with contextlib.redirect_stdout(output_buffer):
                exec(code, namespace)
            result["output"] = output_buffer.getvalue().strip()
            result["success"] = True
        except Exception as e:
            result["output"] = f"Error: {type(e).__name__}: {str(e)}"
            result["success"] = False
    
    thread = threading.Thread(target=run_code)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout)
    
    if thread.is_alive():
        return "Error: Execution timed out", False
    
    return result["output"], result["success"]


def execute_code_blocks_best_effort(code_blocks, namespace, max_blocks=3):
    """
    Execute code blocks with best-effort approach:
    - Only execute up to max_blocks to save time
    - Focus on getting the LAST block to run (verification code)
    
    Returns: (last_output, last_success, num_errors, total_blocks)
    """
    if not code_blocks:
        return "", False, 0, 0
    
    total_blocks = len(code_blocks)
    
    # If too many blocks, only run last few (most relevant)
    blocks_to_run = code_blocks[-max_blocks:] if total_blocks > max_blocks else code_blocks
    
    num_errors = 0
    last_output = ""
    last_success = False
    
    for i, code in enumerate(blocks_to_run):
        output, success = execute_in_namespace(code, namespace, timeout=5)
        
        if not success:
            num_errors += 1
        
        # Always update last output/success for the final block
        if i == len(blocks_to_run) - 1:
            last_output = output
            last_success = success
        elif success and output:
            last_output = output
    
    return last_output, last_success, num_errors, total_blocks


# --- Main Reward Function ---
def sequential_code_reward_func(completions, answer, **kwargs):
    """
    Reward function with STRICT verification + efficiency penalty:
    
    1. PRIMARY: Check \\boxed{} answer in text against GT
    2. SECONDARY: Check code output against GT
    3. EFFICIENCY: Penalize excess code blocks (model looping)
    
    Reward Table:
    ┌─────────────┬──────────────┬─────────────────────────────────┬────────┐
    │ Boxed       │ Code Output  │ Interpretation                  │ Reward │
    ├─────────────┼──────────────┼─────────────────────────────────┼────────┤
    │ Correct     │ Correct (=GT)│ BEST: Properly verified         │ +2.3   │
    │ Correct     │ Wrong (≠GT)  │ BAD: Irrelevant/wrong code      │ +1.8   │
    │ Correct     │ No output    │ Code ran but no print           │ +1.9   │
    │ Correct     │ Crashed      │ Code errored                    │ +1.8   │
    │ Correct     │ No code      │ Didn't verify with code         │ +1.7   │
    │ Wrong       │ Correct (=GT)│ Transcription error (still bad) │ -0.3   │
    │ Wrong       │ Wrong/None   │ Both wrong                      │ -0.5   │
    │ No boxed    │ Any          │ No answer given                 │ -1.5   │
    └─────────────┴──────────────┴─────────────────────────────────┴────────┘
    
    Excess blocks penalty: -0.15 per block beyond 2 (max -0.6)
    """
    rewards = []
    
    for completion, gt_str in zip(completions, answer):
        text = completion[0]["content"]
        
        # Parse ground truth
        try:
            gt_val = float(str(gt_str).replace(',', ''))
        except:
            rewards.append(-2.0)  # Can't parse ground truth
            continue
        
        # PRIMARY: Extract boxed answer from full text output
        boxed_answer = extract_boxed_from_text(text)
        
        # Check if boxed answer is correct
        boxed_correct = False
        if boxed_answer is not None:
            if abs(boxed_answer - gt_val) < 1e-4:
                boxed_correct = True
            elif abs(gt_val) > 1e-6 and abs(boxed_answer - gt_val) / abs(gt_val) < 0.001:
                boxed_correct = True  # 0.1% tolerance
        
        # SECONDARY: Execute code and check output against GT
        code_blocks = extract_all_code_blocks(text)
        num_blocks = len(code_blocks)
        
        code_output_correct = False
        code_ran = False
        code_has_output = False
        code_crashed = False
        
        if code_blocks:
            namespace = create_safe_namespace()
            last_output, last_success, num_errors, _ = execute_code_blocks_best_effort(
                code_blocks, namespace, max_blocks=5
            )
            
            code_ran = True
            code_crashed = not last_success
            
            if last_success and last_output:
                code_has_output = True
                code_val = parse_code_output(last_output)
                if code_val is not None:
                    if abs(code_val - gt_val) < 1e-4:
                        code_output_correct = True
        
        # --- Compute Base Reward ---
        if boxed_answer is None:
            reward = -1.5
        elif boxed_correct:
            if not code_ran:
                reward = 1.7
            elif code_crashed:
                reward = 1.8
            elif not code_has_output:
                reward = 1.9
            elif code_output_correct:
                reward = 2.3  # BEST
            else:
                reward = 1.8  # Irrelevant code
        else:
            # Boxed is WRONG
            if code_output_correct:
                reward = -0.3  # Transcription error
            else:
                reward = -0.5  # Both wrong
        
        # --- Efficiency Penalty for Excess Code Blocks ---
        # Ideal: 1-2 blocks. Penalize looping behavior.
        if num_blocks > 3:
            excess = num_blocks - 3
            penalty = min(excess * 0.15, 0.6)  # Cap at -0.6
            reward -= penalty
        
        rewards.append(reward)
    
    return rewards


# Backward compatible alias
robust_code_reward_func = sequential_code_reward_func

In [20]:
# --- Test / Verification Block ---
# Tests the STRICT verification + efficiency penalty

test_cases = [
    # Case A: BEST - Boxed correct AND code output = GT (1 block)
    {
        "content": "The answer is \\boxed{42}.<|channel|>commentary to=python code<|message|>print(42)<|call|>",
        "ground_truth": "42",
        "expected": 2.3,
        "desc": "Boxed+Code=GT"
    },
    # Case B: Boxed correct BUT code output WRONG
    {
        "content": "Answer is \\boxed{42}.<|channel|>commentary to=python code<|message|>print(999)<|call|>",
        "ground_truth": "42",
        "expected": 1.8,
        "desc": "Boxed OK, Code≠GT"
    },
    # Case C: Boxed correct, no code at all
    {
        "content": "The answer is \\boxed{42}.",
        "ground_truth": "42",
        "expected": 1.7,
        "desc": "Boxed OK, No Code"
    },
    # Case D: Boxed WRONG but code = GT (still penalize boxed!)
    {
        "content": "Answer \\boxed{24}.<|channel|>commentary to=python code<|message|>print(42)<|call|>",
        "ground_truth": "42",
        "expected": -0.3,
        "desc": "Boxed Wrong,Code=GT"
    },
    # Case E: Both wrong
    {
        "content": "\\boxed{100}<|channel|>commentary to=python code<|message|>print(100)<|call|>",
        "ground_truth": "42",
        "expected": -0.5,
        "desc": "Both Wrong"
    },
    # Case F: No boxed
    {
        "content": "The answer is 42.",
        "ground_truth": "42",
        "expected": -1.5,
        "desc": "No Boxed"
    },
    # Case G: EXCESS BLOCKS (3 blocks = 1 extra = -0.15)
    {
        "content": "\\boxed{42}<|channel|>to=python code<|message|>x=1<|call|><|channel|>to=python code<|message|>y=2<|call|><|channel|>to=python code<|message|>print(42)<|call|>",
        "ground_truth": "42",
        "expected": 2.15,  # 2.3 - 0.15
        "desc": "3 Blocks (-0.15)"
    },
    # Case H: MANY EXCESS BLOCKS (5 blocks = 3 extra = -0.45)
    {
        "content": "\\boxed{42}<|channel|>to=python code<|message|>a=1<|call|><|channel|>to=python code<|message|>b=2<|call|><|channel|>to=python code<|message|>c=3<|call|><|channel|>to=python code<|message|>d=4<|call|><|channel|>to=python code<|message|>print(42)<|call|>",
        "ground_truth": "42",
        "expected": 1.85,  # 2.3 - 0.45
        "desc": "5 Blocks (-0.45)"
    },
    # Case I: Markdown format
    {
        "content": "\\boxed{42}\n```python\nprint(42)\n```",
        "ground_truth": "42",
        "expected": 2.3,
        "desc": "Markdown OK"
    },
]

# Format for the Reward Function
completions = [[{"content": t["content"]}] for t in test_cases]
answers = [t["ground_truth"] for t in test_cases]

# Run the Reward Function
print(f"{'Description':<18} | {'Exp':<5} | {'Got':<5} | {'Status'}")
print("-" * 55)

results = sequential_code_reward_func(completions, answers)

all_pass = True
for i, score in enumerate(results):
    expected = test_cases[i]["expected"]
    status = "✅ PASS" if abs(score - expected) < 0.05 else "❌ FAIL"
    if "FAIL" in status:
        all_pass = False
    print(f"{test_cases[i]['desc']:<18} | {expected:<5} | {score:<5.2f} | {status}")

print("-" * 55)
print(f"Overall: {'✅ ALL TESTS PASSED' if all_pass else '❌ SOME TESTS FAILED'}")

Description        | Exp   | Got   | Status
-------------------------------------------------------
Boxed+Code=GT      | 2.3   | 2.30  | ✅ PASS
Boxed OK, Code≠GT  | 1.8   | 1.80  | ✅ PASS
Boxed OK, No Code  | 1.7   | 1.70  | ✅ PASS
Boxed Wrong,Code=GT | -0.3  | -0.30 | ✅ PASS
Both Wrong         | -0.5  | -0.50 | ✅ PASS
No Boxed           | -1.5  | -1.50 | ✅ PASS
3 Blocks (-0.15)   | 2.15  | 2.30  | ❌ FAIL
5 Blocks (-0.45)   | 1.85  | 2.00  | ❌ FAIL
Markdown OK        | 2.3   | 2.30  | ✅ PASS
-------------------------------------------------------
Overall: ❌ SOME TESTS FAILED


<h1 style="color:  #e323a9ff;">🗃️ Dataset</h1>

<h2 style="color: #e323a9ff;">🔁 Replicate Samples for More Steps</h2>

<div style="padding: 10px 12px; border-left: 6px solid #58d987ff; background: #F0FDF4; border-radius: 10px;">
Creates a Hugging Face <b>Dataset</b> and replicates entries so GRPO can run for many steps 🧩
</div>

In [21]:
# Create the HuggingFace Dataset
from datasets import Dataset

# Replicate the 50 problems to have enough data for 100 steps
# With batch_size=1 and 100 steps, we need at least 100 samples
replicated_data =  train_data * 1  # 50 problems * 2 = 100 samples

dataset = Dataset.from_list(replicated_data)

# Calculate prompt length for configuration
sample_prompt = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort="medium"
)
max_prompt_length = len(tokenizer(sample_prompt)["input_ids"]) + 10  # Add buffer

print(f"Dataset size: {len(dataset)}")
print(f"Max prompt length: {max_prompt_length}")
print(f"Sample formatted prompt:\n{sample_prompt[:500]}...")

Dataset size: 2
Max prompt length: 278
Sample formatted prompt:
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-21

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Let $f(x): [0, 1] \rightarrow \mathbb{R}$, satisfying: (1) $f(\frac{x}{3}) = \frac{1}{2}f(x)$; (2) $f(1-x) = 1 - f(x)$; (3) $f(x) = \frac{1}{2} (x \in [\f...


<h1 style="color:  #e323a9ff;">🧪 GRPO</h1>

<h2 style="color: #e323a9ff;">⚙️ need to understand GRPO</h2>

<div style="padding: 10px 12px; border-left: 6px solid #3B82F6; background: #EFF6FF; border-radius: 10px;">
Sets sampling + optimization hyperparams.<br>
Watch <b>max_prompt_length</b> + <b>max_completion_length</b> to avoid truncation ✂️
</div>

In [22]:
# !uv pip install wandb 

In [23]:
# import wandb
# wandb.login(key="YOUR_WANDB_API_KEY")

# run = wandb.init(
#     entity="barnoahmed666-none",
#     project="OSS GRPO ",
#     config={
#         "model_name": "unsloth/gpt-oss-20b",
#         "lora_rank": lora_rank,
#         "max_seq_length": max_seq_length,
#         "learning_rate": 5e-5,
#         "weight_decay": 0.001,
#         "warmup_ratio": 0.1,
#         "lr_scheduler_type": "linear",
#         "optim": "adamw_8bit",
#         "per_device_train_batch_size": 2,
#         "gradient_accumulation_steps": 1,
#         "num_generations": 2,
#         "max_steps": 100,
#         "temperature": 1.0,
#     },
# )


In [24]:
# Configure GRPO Training
from trl import GRPOConfig, GRPOTrainer
import gc

# Cap completion length to something reasonable for speed
max_completion_length = max_seq_length - max_prompt_length    #min(max_seq_length - max_prompt_length, 512)  # Cap at 512 tokens

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    num_generations = 2 ,  # Number of completions per prompt
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 2,  # 100 steps for testing
    #save_steps = 50,
    report_to = "none",
    output_dir = "outputs_grpo_test",
    #beta=0.01,
)

print(f"Max completion length: {max_completion_length}")
print("Training config ready!")

Max completion length: 16106
Training config ready!


<h2 style="color: #e323a9ff;">🏗️ Build the Trainer</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #F5F3FF; border-radius: 10px;">
Wires together: <b>model</b> + <b>tokenizer</b> + <b>reward functions</b> + <b>dataset</b> 🧷
</div>

In [25]:
# Initialize the GRPO Trainer
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        robust_code_reward_func
       # format_reward,    # Reward for using \boxed{}
       # answer_reward,    # Main reward: distance-based correctness
    ],
    args = training_args,
    train_dataset = dataset,
)

print("Trainer initialized!")

Trainer initialized!


<h1 style="color: #e323a9ff;">🚀 Training Run</h1>

<h2 style="color: #e323a9ff;">🏋️ GRPO Train</h2>

<div style="padding: 10px 12px; border-left: 6px solid #EF4444; background: #FEF2F2; border-radius: 10px;">
Heads up: generation dominates runtime ⏳<br>
If it’s too slow, reduce <b>max_steps</b>, <b>num_generations</b>, or <b>max_new_tokens</b>.
</div>

In [26]:
# Start training - 100 steps
# Monitor the 'reward' column in the output table - it should increase over time
import time

print("Starting training... (This will take a while - generation is the slow part)")
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time
print(f"\n{'='*50}")
print(f"Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Time per step: {training_time/100:.2f} seconds")
print(f"Estimated time for 1000 steps: {(training_time/100)*1000/60:.2f} minutes")

Starting training... (This will take a while - generation is the slow part)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 1 | Total steps = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 11,943,936 of 116,841,100,608 (0.01% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / sequential_code_reward_func / mean,rewards / sequential_code_reward_func / std
1,0.000000,1.700000,0.000000,2718.500000,2305.000000,3132.000000,0.000000,2718.500000,2305.000000,3132.000000,0,0,0,0,0,0.002435,1.700000,0.000000
2,0.000000,-2.000000,0.000000,11883.000000,7969.000000,15797.000000,0.000000,11883.000000,7969.000000,15797.000000,No Log,No Log,No Log,No Log,No Log,0.002437,-2.000000,0.000000


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



Training completed in 22766.26 seconds (379.44 minutes)
Time per step: 227.66 seconds
Estimated time for 1000 steps: 3794.38 minutes


<h1 style="color: #e323a9ff;">🔎 Evaluation</h1>

<h2 style="color: #e323a9ff;">🧠 Quick Inference Smoke Test</h2>

<div style="padding: 10px 12px; border-left: 6px solid #10B981; background: #ECFDF5; border-radius: 10px;">
Generates on a known training prompt and prints the model output 🧪
</div>

In [27]:
# Switch to eval mode (faster inference, disables gradient checkpointing)
model.eval()

# Test inference after training
text = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "medium",
)

from transformers import TextStreamer

print("Testing trained model on first problem:")
print("="*50)
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)
print(f"\nExpected answer: {train_data[0]['answer']}")

Testing trained model on first problem:
<|channel|>analysis<|message|>We need to understand function definition. There is condition (3): $f(x) = \frac{1}{2} (x \in [\frac{1}{3}, \frac{2}{3}])$. I think they mean that f(x) equals 1/2 when x in [1/3,2/3]; otherwise defined by functional equations? Probably piecewise defined: For any x, we can recursively apply rules (1) and (2) to reduce to base interval? This resembles Cantor function? Indeed functional equations reminiscent of Cantor function: f(x/3) = f(x)/2, f(1 - x) = 1 - f(x). plus middle interval constant 1/2.

Thus f is Cantor function mapping Cantor set to [0,1]. Indeed the standard Cantor function satisfies f(x/3)=f(x)/2, f((2+x)/3)= (1+f(x))/2 maybe. But we have symmetry.

Given that, we need sum over odd k from 1 to 3^n of f(k/3^n). For n=2023, huge. Need combinatorial pattern. Since odd k corresponds to ternary expansion with last digit 1 or 2? Actually denominator 3^n. Points k/3^n with k integer from 1 to 3^n. These are dy

In [28]:
# Save the LoRA adapters
model.save_pretrained("gpt_oss_120b_math_lora_test")
tokenizer.save_pretrained("gpt_oss_120b_math_lora_test")
print("LoRA adapters saved to 'gpt_oss_20b_math_lora_test'")

LoRA adapters saved to 'gpt_oss_20b_math_lora_test'


## Optional: Save merged model or push to Hub

Uncomment the options below as needed:
- **LoRA only**: Smallest size, requires base model to load
- **Merged 16bit**: Full model in fp16
- **MXFP4**: GPT-OSS native precision, good for VLLM

In [29]:
# Optional: Merge and save in different formats

# Save merged model in MXFP4 (GPT-OSS native precision)
# model.save_pretrained_merged("gpt_oss_20b_math_mxfp4", tokenizer, save_method="mxfp4")

# Save merged model in 16bit
# model.save_pretrained_merged("gpt_oss_20b_math_16bit", tokenizer, save_method="merged_16bit")

# Push to Hugging Face Hub (uncomment and add your token)
# model.push_to_hub_merged("your-username/gpt-oss-20b-math", tokenizer, token="hf_...", save_method="lora")